In [ ]:
# 01 · STATE STAGE ACT 단일 정책 CONFIG · GitHub 중단 복구 · Drive 미사용
CFG = {
    # 저장 · ver2는 기존 DP 결과와 별도 Release에 저장
    'run_name': 'moveboxes_stage_chunk_deadline_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': 'stage-act-chunk-compare',
    'output_root': '/content/moveboxes_runs',

    # 데이터 / Colab 2026.07 · Python 3.12 · T4
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 학습 · 시뮬레이터 없이 CPU 데이터 + GPU 모델
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'total_iters': {'easy': 12000, 'medium': 20000, 'hard': 30000},
    'amp': True,

    # 작은 State ACT · 기존 DP 체크포인트 사용 불가
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 검증 / 중단 복구 / 작은 관측 위치 증강
    'save_freq': 1000,
    'warmup_steps': 500,
    'validation_batches': 8,
    'kl_weight': 0.001,
    'position_noise': 0.001,

    # 실행 · 매 스텝 재계획, 최근 XYZ 예측 평균, 집게는 최신 예측
    'temporal_decay': 0.25,
    'ensemble_window': 4,
    'ensemble_candidates': [1, 4],

    # 빠른 테스트 / 최종 평가 · 시드 분리, 기존 200스텝 유지
    'test_episodes': 8,
    'test_seed_start': 40000,
    'test_record_video': True,
    'tuning_episodes': 8,
    'tuning_seed_start': 20000,
    'benchmark_episodes': 100,
    'eval_seed_start': 30000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'record_eval_video': True,

    # 출력
    'console_interval_seconds': 10,
    'team': 'my-team',

    # 실행 조건으로 행동 학습 · 빠른 테스트가 0이면 긴 평가 생략
    'action_training_mode': 'prior',
    'repair_iters': 2000,
    'allow_zero_success_evaluation': False,

    # 단계 판단 · 학습된 완료/복구 확신이 낮으면 현재 단계 유지
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'stage_loss_weight': 0.3,
    'gate_loss_weight': 0.3,

    # 복구 시연 · 수집 전용 expert, 학습/제출은 학습된 정책
    'recovery_episodes': 16,
    'recovery_max_attempts': 48,
    'recovery_seed_start': 100000,
    'collection_max_steps': {'easy': 500, 'medium': 900, 'hard': 1400},
    'noise_probability': 0.08,
    'action_noise_std': 0.12,
    'drop_probability': 0.015,

}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from stage_experiment import StageExperiment, source_bundle
experiment = StageExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


## 실행 순서

03 셀은 `GH_TOKEN` 환경변수, Colab Secrets, 비공개 입력창 순서로 GitHub 토큰을 받습니다. 같은 `run_name`의 **현재 학습 상태만** GitHub Release에서 복원합니다. 과거 best checkpoint를 자동으로 가져오지 않습니다.

04~05에서 T4 환경과 state dataset을 준비합니다. 각 난이도는 recovery 수집 후 Stage ACT를 학습합니다. 수집은 매 시도, 학습은 매 1,000 iteration마다 GitHub에 동기화됩니다. 런타임이 끊기면 새 런타임에서 01~05와 해당 난이도의 수집·학습 셀을 다시 실행하면 이어집니다.


In [ ]:
# 07 · EASY recovery 수집 · 매 시도 GitHub 저장, 재실행 시 이어서 진행
experiment.collect('easy')


In [ ]:
# 08 · EASY State Stage ACT 학습 · 로그 표시 + 매 checkpoint 완전 복구 저장
# latest.pt에는 model, optimizer, AMP scaler, sampling/CPU/CUDA RNG가 함께 저장됩니다.
experiment.train('easy')


In [ ]:
# 09 · MEDIUM recovery 수집 · 매 시도 GitHub 저장, 재실행 시 이어서 진행
experiment.collect('medium')


In [ ]:
# 10 · MEDIUM State Stage ACT 학습 · 로그 표시 + 매 checkpoint 완전 복구 저장
# latest.pt에는 model, optimizer, AMP scaler, sampling/CPU/CUDA RNG가 함께 저장됩니다.
experiment.train('medium')


In [ ]:
# 11 · HARD recovery 수집 · 매 시도 GitHub 저장, 재실행 시 이어서 진행
experiment.collect('hard')


In [ ]:
# 12 · HARD State Stage ACT 학습 · 로그 표시 + 매 checkpoint 완전 복구 저장
# latest.pt에는 model, optimizer, AMP scaler, sampling/CPU/CUDA RNG가 함께 저장됩니다.
experiment.train('hard')


In [ ]:
# 현재 run의 학습 checkpoint에 단일 stage-aware 실행 정책 적용
import hashlib, importlib.metadata, os, shutil, subprocess, sys
from pathlib import Path
import torch

RUN_DIR = Path(CFG['output_root'])/CFG['run_name']
CANDIDATE = RUN_DIR/'integrated_candidate'
CHECKPOINT_OVERRIDES = {'easy':'', 'medium':'', 'hard':''}
STAGE_HORIZONS = dict(pick=2, carry=6, place=2, done=1)
GRIPPER_MARGIN = .5
GRIPPER_CONFIRM_STEPS = 2
MAX_STEPS = 200
SMOKE_SEED = 61000
DIMS = {'easy':54, 'medium':72, 'hard':90}

# 경로를 따로 지정하지 않으면 이 run에서 학습된 best_val/latest만 사용합니다.
selected = {}
for level in DIMS:
    override = CHECKPOINT_OVERRIDES[level]
    if override:
        checkpoint = Path(override).expanduser().resolve()
    else:
        folder = RUN_DIR/level/'checkpoints'
        checkpoint = next((p for p in (folder/'best_val.pt', folder/'latest.pt') if p.is_file()), None)
    if checkpoint is not None:
        selected[level] = checkpoint
if not selected:
    raise FileNotFoundError('먼저 하나 이상의 experiment.train(level) 셀을 실행하세요.')

def sha256(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

CANDIDATE.mkdir(parents=True, exist_ok=True)
for source in ('stage_chunk_policy.py','stage_policy.py','stage_model.py','stage_schema.py'):
    shutil.copy2(PROJECT/'ver2/stages'/source, CANDIDATE/source)
shutil.copy2(PROJECT/'ver2/act_v2_model.py', CANDIDATE/'act_v2_model.py')

manifest = dict(policy='state_stage_act_chunk_fsm', project_commit=CFG['project_commit'],
                official_commit=CFG['repo_commit'], run_name=CFG['run_name'], levels={})
level_lines = []
for level, checkpoint in selected.items():
    saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
    if saved.get('format') != 'moveboxes-stage-act-v1':
        raise ValueError(f'{level}: Stage ACT checkpoint가 아닙니다: {saved.get("format")}')
    model_config = saved['model_config']
    if model_config.get('state_dim') != DIMS[level] or model_config.get('chunk_size', 0) < 6:
        raise ValueError(f'{level}: state dimension 또는 trained chunk size 불일치')
    sidecar = checkpoint.parent/'policy_config.json'
    if not sidecar.is_file():
        raise FileNotFoundError(sidecar)
    policy = json.loads(sidecar.read_text(encoding='utf-8'))
    if policy.get('model_config') != model_config:
        raise ValueError(f'{level}: checkpoint/sidecar architecture 불일치')
    policy.update(model_config=model_config, stage_aware_chunk=True,
                  stage_horizons=STAGE_HORIZONS, gripper_fsm=True,
                  gripper_margin=GRIPPER_MARGIN,
                  gripper_confirm_steps=GRIPPER_CONFIRM_STEPS,
                  auto_reset_steps=MAX_STEPS-1)
    policy.pop('act_horizon', None)
    policy.pop('num_inference_steps', None)
    target_dir = CANDIDATE/'checkpoints'/level
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir/'model.pt'
    shutil.copy2(checkpoint, target)
    (target_dir/'policy_config.json').write_text(json.dumps(policy, indent=2), encoding='utf-8')
    if sha256(target) != sha256(checkpoint):
        raise RuntimeError(f'{level}: checkpoint 사본 해시 불일치')
    manifest['levels'][level] = dict(source=str(checkpoint), checkpoint_sha256=sha256(target),
                                     step=saved.get('step'), model_config=model_config,
                                     policy_config=policy)
    level_lines.append(f'    {level}: {{ checkpoint: checkpoints/{level}/model.pt }}')

submission = 'team: '+json.dumps(CFG['team'])+'\nstate:\n  policy: stage_chunk_policy:load_policy\n  levels:\n'+'\n'.join(level_lines)+'\n'
(CANDIDATE/'submission.yaml').write_text(submission, encoding='utf-8')
(CANDIDATE/'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('현재 run checkpoint:', {k:str(v) for k,v in selected.items()})
print('단일 integrated candidate:', CANDIDATE)


In [ ]:
# 실제 simulator state/action/buffer/reset sanity check
from stage_chunk_policy import load_policy
from warehouse_sort.utils import compose_cfg, make_env

for level in selected:
    cfg = compose_cfg(['difficulty='+level, 'obs_mode=state', 'max_episode_steps='+str(MAX_STEPS)],
                      config_dir=str(Path(CFG['repo_dir'])/'conf'))
    env, _ = make_env(cfg, 'state', cfg.randomization, num_envs=1)
    try:
        obs, _ = env.reset(seed=SMOKE_SEED)
        assert tuple(obs.shape) == (1, DIMS[level])
        agent = load_policy(CANDIDATE/'checkpoints'/level/'model.pt', obs,
                            env.single_action_space, 'cuda')
        with torch.no_grad():
            first = agent.act(obs)
        assert first.shape == (1,4) and torch.isfinite(first).all()
        assert first.abs().max() <= 1 and first[0,3].item() in (-1.,1.)
        assert agent.action_buffer.shape[1] == manifest['levels'][level]['model_config']['chunk_size']
        agent.reset()
        assert not agent.history and agent.action_buffer is None and agent.grip_state is None
        obs2, _ = env.reset(seed=SMOKE_SEED)
        with torch.no_grad():
            repeated = agent.act(obs2)
        torch.testing.assert_close(first, repeated, rtol=0, atol=0)
        agent.step = MAX_STEPS-1
        agent.history.append(torch.full_like(obs2, 123.))
        agent.action_buffer.fill_(123.)
        with torch.no_grad():
            boundary = agent.act(obs2)
        torch.testing.assert_close(first, boundary, rtol=0, atol=0)
        assert agent.step == 1
        env.step(repeated)
        print(level, 'state/action/buffer/manual+official reset/env.step OK')
    finally:
        env.close()


In [ ]:
# 공식 eval.py · 1 episode smoke
RESULTS = RUN_DIR
SMOKE_CONFIG = RUN_DIR/'integrated_smoke_eval.yaml'
SMOKE_CONFIG.write_text('eval:\n  n_episodes: 1\n  seeds: ['+str(SMOKE_SEED)+']\n', encoding='utf-8')
UPSTREAM = Path(CFG['repo_dir'])

def run_official(level, eval_config, label):
    output = RUN_DIR/level/'integrated_official_eval'/label
    output.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(UPSTREAM/'eval.py'), 'difficulty='+level,
        'obs_mode=state', 'policy=stage_chunk_policy:load_policy',
        'checkpoint='+str(CANDIDATE/'checkpoints'/level/'model.pt'),
        'eval_config='+str(eval_config), 'max_episode_steps='+str(MAX_STEPS),
        'hydra.run.dir='+str(output)]
    child_env = dict(os.environ)
    child_env['PYTHONPATH'] = str(CANDIDATE)+os.pathsep+str(UPSTREAM)+os.pathsep+child_env.get('PYTHONPATH','')
    log = output/'official_eval.log'
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=UPSTREAM, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, errors='replace', bufsize=1)
        for line in process.stdout:
            print(line, end='')
            handle.write(line)
            handle.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f'official eval failed ({level}); {log} 확인')
    experiment.sync_level(level)
    return log

for level in selected:
    run_official(level, SMOKE_CONFIG, 'smoke')


In [ ]:
# 공식 conf/eval/default.yaml 평가 · 자체 metric/승자 선택 없음
OFFICIAL_EVAL_CONFIG = UPSTREAM/'conf/eval/default.yaml'
if not OFFICIAL_EVAL_CONFIG.is_file():
    raise FileNotFoundError(OFFICIAL_EVAL_CONFIG)
for level in selected:
    run_official(level, OFFICIAL_EVAL_CONFIG, 'default')
print('공식 raw logs/videos:', {level:str(RUN_DIR/level/'integrated_official_eval'/'default') for level in selected})


In [ ]:
# 단일 candidate ZIP 생성 및 브라우저 다운로드
check = """import json,sys,torch
from pathlib import Path
from types import SimpleNamespace
from stage_chunk_policy import load_policy
root=Path(sys.argv[1])
manifest=json.loads((root/'manifest.json').read_text())
for level,row in manifest['levels'].items():
 p=root/'checkpoints'/level/'model.pt'
 agent=load_policy(p,torch.zeros(1,row['model_config']['state_dim']),SimpleNamespace(shape=(4,)),'cpu')
 a=agent.act(torch.zeros(1,row['model_config']['state_dim']))
 assert a.shape==(1,4) and torch.isfinite(a).all() and a.abs().max()<=1
 agent.reset()
 print(level,'candidate import/action/reset OK')
"""
subprocess.run([sys.executable, '-c', check, str(CANDIDATE)], cwd=CANDIDATE, check=True)
archive = shutil.make_archive(str(CANDIDATE), 'zip', CANDIDATE)
from google.colab import files
print('candidate ZIP:', archive)
files.download(archive)
